<a href="https://colab.research.google.com/github/Vysokodelovoi/IAD_GP/blob/main/gp_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Построение моделей машинного обучения

In [ ]:
!pip install optuna

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import catboost as cb
from catboost import CatBoostClassifier
import optuna
import ml_utils as mu

In [ ]:
df = pd.read_parquet('main_df_edited_no_dup.parquet')

In [ ]:
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,country_full,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,0,Transient,98.00,0,1,Check-Out,2015-07-03,United Kingdom,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,0,Transient,96.14,0,0,Check-Out,2017-09-06,Belgium,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,0,Transient,225.43,0,2,Check-Out,2017-09-07,France,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,157.71,0,4,Check-Out,2017-09-07,Germany,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,104.40,0,0,Check-Out,2017-09-07,United Kingdom,7


In [ ]:
# Линейные модели logistic_regression / linear, Даниэль
#  svm Игорь
#  boosting Игорь
#  random trees Даниэль
#  нейронка Игорь
#  изотоническая регрессия Даниэль
#  ранжирование попробовать? Игорь

#  Для каждой затюнить гиперпараметры через GridSearchCV или optuna
#  Результаты по проведенным экспериментам собрать в pandas датасет и отправить
#  Сделать выводы по performance моделю, посмотреть на важность признаков через shap или feature_importances


#  Графики с дубликатами и без сравнить

In [ ]:
df_model = df[[c for c in df.columns if c not in ['reservation_status', 'reservation_status_date', 'country_full']]]

In [ ]:
df_model

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,C,C,3,No Deposit,0,Transient,0.00,0,0,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,C,C,4,No Deposit,0,Transient,0.00,0,0,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,A,C,0,No Deposit,0,Transient,75.00,0,0,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,A,A,0,No Deposit,0,Transient,75.00,0,0,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,A,0,No Deposit,0,Transient,98.00,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,A,A,0,No Deposit,0,Transient,96.14,0,0,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,E,E,0,No Deposit,0,Transient,225.43,0,2,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,D,D,0,No Deposit,0,Transient,157.71,0,4,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,A,A,0,No Deposit,0,Transient,104.40,0,0,7


In [ ]:
X = df_model[[c for c in df_model.columns if c != 'is_canceled']].copy()
y = df_model.is_canceled.copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

## SVM

In [ ]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

In [ ]:
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA()),
    ('classifier', LinearSVC(dual=False, random_state=8)),
])

In [ ]:
def objective(trial):

    max_iter_param = trial.suggest_int('classifier__max_iter', 1000, 10000, step=1000)
    tol_param = trial.suggest_float('classifier__tol', 1e-5, 1e-2, log=True)
    c_param = trial.suggest_float('classifier__C', 1e-4, 1e2, log=True)

    scaler_param = trial.suggest_categorical('scaler_type', ['standard', 'robust'])
    scaler = StandardScaler() if scaler_param == 'standard' else RobustScaler()

    use_pca = trial.suggest_categorical('use_pca', [True, False])

    if use_pca:
        pca_components = trial.suggest_int('pca__n_components', 1, 35)
        pca_step = PCA(n_components=pca_components)
    else:
        pca_step = 'passthrough'

    svm_pipeline.set_params(
        preprocessor__scaler=scaler,
        pca=pca_step,
        classifier__C=c_param,
        classifier__max_iter=max_iter_param,
        classifier__tol=tol_param
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
    scores = cross_val_score(
        svm_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, show_progress_bar=True, n_trials=50)

# print("Лучшие параметры:", study.best_params)
# print("Лучший F1-score:", study.best_value)

In [ ]:
# best = study.best_params

In [ ]:
best = {'classifier__max_iter': 10000, 'classifier__tol': 0.0008743508260033666, 'classifier__C': 36.856060983954215, 'scaler_type': 'standard', 'use_pca': False}

best_scaler = StandardScaler() if best['scaler_type'] == 'standard' else RobustScaler()

if best.get('use_pca', False):
    best_pca = PCA(n_components=best['pca__n_components'])
else:
    best_pca = 'passthrough'

svm_pipeline.set_params(
    preprocessor__scaler=best_scaler,
    pca=best_pca,
    classifier__C=best['classifier__C'],
    classifier__max_iter=best['classifier__max_iter'],
    classifier__tol=best['classifier__tol']
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'arrival_date_week_number',
                                                   'arrival_date_day_of_month',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'adults', 'children',
                                                   'babies',
                                                   'is_repeated_guest',
                                                   'previous_cancellations',
                                                   'previous_bookings_not_canceled',
                                                   'booking_change...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['hotel',
                                                   'arrival_date_month', 'meal',
                                                   'country', 'market_segment',
                                                   'distribution_channel',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_type',
                                                   'customer_type'])])),
                ('pca', 'passthrough'),
                ('classifier',
                 LinearSVC(C=36.856060983954215, dual=False, max_iter=10000,
                           random_state=8, tol=0.0008743508260033666))])

In [ ]:
import importlib
import ml_utils as mu

# Эта команда принудительно обновит модуль в памяти Jupyter
importlib.reload(mu)

<module 'ml_utils' from '/content/ml_utils.py'>

In [ ]:
mu.run_experiment(svm_pipeline, 'svm', 'main_df_edited_no_dup.parquet', best)

svm | score: 0.55317 time: 5.7s


/content/ml_utils.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([result_row])], ignore_index=True)


{'timestamp': '2026-06-14T22:58:17.660727',
 'method': 'svm',
 'params': '{"classifier__max_iter": 10000, "classifier__tol": 0.0008743508260033666, "classifier__C": 36.856060983954215, "scaler_type": "standard", "use_pca": false}',
 'score': 0.553168880455408,
 'duration_sec': 5.713756799697876}

## CatBoost

In [ ]:
X = df_model[[c for c in df_model.columns if c not in ['is_canceled','country_full']]].copy()
y = df_model.is_canceled.copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

In [ ]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000, step=100),
        'depth': trial.suggest_int('depth', 4, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'cat_features': cat_features,
        'random_seed': 8,
        'thread_count': -1,
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 100, step=5)
    }

    model = CatBoostClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):

        X_train_train = X_train.iloc[train_idx]
        X_train_val = X_train.iloc[val_idx]
        y_train_train = y_train.iloc[train_idx]
        y_train_val = y_train.iloc[val_idx]

        model = cb.CatBoostClassifier(**params)

        model.fit(
            X_train_train, y_train_train,
            eval_set=(X_train_val, y_train_val),
            verbose=False
        )

        y_pred = model.predict(X_train_val)
        scores.append(f1_score(y_train_val, y_pred))

    return np.mean(scores)

In [ ]:
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2026-06-14 16:30:45,557] A new study created in memory with name: no-name-78b32a70-b284-4478-9fc9-28277e4a28d8


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-14 16:38:29,971] Trial 0 finished with value: 0.6886515832289016 and parameters: {'iterations': 1000, 'depth': 4, 'learning_rate': 0.05525117752762227, 'l2_leaf_reg': 0.9709957380220429, 'early_stopping_rounds': 85}. Best is trial 0 with value: 0.6886515832289016.
[I 2026-06-14 16:43:44,086] Trial 1 finished with value: 0.6787954117764385 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.03656176676313291, 'l2_leaf_reg': 0.05305348598636522, 'early_stopping_rounds': 80}. Best is trial 0 with value: 0.6886515832289016.
[I 2026-06-14 16:50:41,822] Trial 2 finished with value: 0.6980778170816262 and parameters: {'iterations': 900, 'depth': 4, 'learning_rate': 0.15179953122208972, 'l2_leaf_reg': 0.02629742319435691, 'early_stopping_rounds': 85}. Best is trial 2 with value: 0.6980778170816262.
[I 2026-06-14 16:56:52,358] Trial 3 finished with value: 0.6930103966573767 and parameters: {'iterations': 1000, 'depth': 5, 'learning_rate': 0.06900829323089025, 'l2_leaf_

KeyboardInterrupt: 

In [ ]:
best = study_cb.best_params
stats_catboost = pd.DataFrame(best, index=['catboost'])

In [ ]:
stats = pd.concat([stats, stats_catboost], axis=0)

In [ ]:
stats

In [ ]:
#